# Tracing with mlflow

## Environment Set-up

Create a virtual environment:

```
python -m venv .venv
```

Activate the virtual environment:

```
source .venv/bin/activate
```

Install the dependencies

```
pip install -r requirements.txt
```

Start mlflow server:

```
mlflow server --host 0.0.0.0 --port 2000
```

In [1]:
# Connect to mlflow server and set experiment

import mlflow

mlflow.set_tracking_uri("http://localhost:2000")
mlflow.set_experiment("Tracing Quickstart")

<Experiment: artifact_location='mlflow-artifacts:/641425381063492368', creation_time=1759892577179, experiment_id='641425381063492368', last_update_time=1759892577179, lifecycle_stage='active', name='Tracing Quickstart', tags={'mlflow.experimentKind': 'genai_development'}>

Go to http://localhost:2000 to access the mlflow UI

In [2]:
# Load environment variables

import dotenv

dotenv.load_dotenv()

True

## Automatic tracing

Easy one-line tracing

In [3]:
# Tracing with OpenAI

import mlflow
from openai import OpenAI

mlflow.openai.autolog() # One-liner tracing

client = OpenAI()

client.chat.completions.create(
    model="gpt-4.1-nano",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France?"},
    ],
)

2025/10/09 12:14:35 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}


ChatCompletion(id='chatcmpl-COZWlLWaeKEH6yu0KkYC5zZdJTEyG', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The capital of France is Paris.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1759972475, model='gpt-4.1-nano-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint='fp_7c233bf9d1', usage=CompletionUsage(completion_tokens=7, prompt_tokens=24, total_tokens=31, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

Trace(trace_id=tr-10c0c91f657047c367920faae9e04e09)

## Manual tracing

If you want to trace more complex workflows, mlflow exposes the very useful `@mlflow.trace` decorator which you can use to trace any function.

In [4]:
@mlflow.trace(name="add",)
def add(x, y):
    return x + y

In [5]:
add(1, 2)

2025/10/09 12:14:35 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}


3

Trace(trace_id=tr-e835837dc99079163950cd07f9c80ef6)

## Agentic workflow + Tracing

Let's look at a more complicated workflow which will involve function calling. We will trace this tool.

In [6]:
import requests
from mlflow.entities import SpanType

# Add decorator to trace the tool
@mlflow.trace(span_type=SpanType.TOOL)
def get_weather(latitude, longitude):
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m"
    )
    data = response.json()
    return data["current"]["temperature_2m"]

In [7]:
# Define OpenAI tools

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current temperature for provided coordinates in celsius.",
            "parameters": {
                "type": "object",
                "properties": {
                    "latitude": {"type": "number"},
                    "longitude": {"type": "number"},
                },
                "required": ["latitude", "longitude"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    }
]

In [8]:
# Create a function to run the agentic workflow

import json
from mlflow.entities import SpanType

@mlflow.trace(span_type=SpanType.AGENT)
def run_weather_agent(question: str):
    messages = [
        {"role": "user", "content": question}]

    response = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=messages,
        tools=tools,
    )
    ai_msg = response.choices[0].message
    messages.append(ai_msg)

    if tool_calls := ai_msg.tool_calls:
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            if function_name == "get_weather":
                kwargs = json.loads(tool_call.function.arguments)
                tool_result = get_weather(**kwargs)
            else:
                raise RuntimeError("An invalid tool is returned from the LLM")
            
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": str(tool_result)
                }
            )

        response = client.chat.completions.create(model="o4-mini", messages=messages)

    return response.choices[0].message.content
            

In [9]:
# Run the agentic workflow

question = "What is the weather in Melbourne?"
run_weather_agent(question)

2025/10/09 12:14:37 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/09 12:14:38 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/09 12:14:40 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/09 12:14:40 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}


'The current temperature in Melbourne is 18.8°C. If you need more detailed weather information—like humidity, wind speed, or a forecast—just let me know!'

Trace(trace_id=tr-7eb54083839564de8ae5a604373f3eb5)

## SpanType

[SpanType API Reference](https://mlflow.org/docs/3.2.0/api_reference/python_api/mlflow.entities.html?highlight=spantype#mlflow.entities.SpanType)

In [10]:
import anthropic

ant = anthropic.Anthropic()

In [11]:
# Combining manual tracing and automatic tracing

mlflow.anthropic.autolog()

@mlflow.trace(span_type=SpanType.CHAIN)
def run_chain(query: str):
    messages = build_messages(query)

    # Anthropic auto tracing will take care of this call
    response = ant.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=1024,
        messages=messages,
    )

    return parse_response(response)

@mlflow.trace
def build_messages(query: str):
    return [
        {"role": "user", "content": query},
    ]

@mlflow.trace
def parse_response(response):
    return response.content[0].text

run_chain("What is the study of fluid dynamics?")

2025/10/09 12:14:40 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/09 12:14:44 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/09 12:14:44 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/09 12:14:44 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}


'Fluid dynamics is a branch of physics that deals with the study of fluids (liquids and gases) in motion. It focuses on understanding how fluids behave when they flow and interact with surfaces, other fluids, or solid objects. This field involves analyzing various properties such as:\n\n1. Fluid flow characteristics\n2. Pressure\n3. Velocity\n4. Density\n5. Temperature\n6. Viscosity\n\nFluid dynamics has numerous practical applications, including:\n- Aerodynamics in aircraft and automotive design\n- Weather prediction\n- Blood flow in medical research\n- Ocean currents\n- Hydraulic engineering\n- Combustion processes\n- Computational modeling\n\nScientists and engineers use mathematical equations and computer simulations to understand and predict fluid behavior in different scenarios.\n\nThe field combines principles from physics, mathematics, and engineering to explain and predict complex fluid movement and interactions.'

Trace(trace_id=tr-248b849a25df6bbdfdf6eb3f01b5b1d7)

# Complex agent

Let's build a complex agent which does tool calls, uses langchain auto-tracing, and has access to a vector database - adding tracing to everything.

In [12]:
mlflow.langchain.autolog()

In [13]:
# Set-up vector database

import chromadb

chroma_client = chromadb.Client()

collection = chroma_client.create_collection("documents")

collection.add(
    ids=["id1", "id2", "id3"],
    documents=[
        "Travel time: One-way flight from Vandul to Farish is 1 hour 30 minutes. One-way flight Equoza to Opinstian is 3 hours 45 minutes. One-way flight from Maddox to Lenivin is 8 hours.",
        "Essential packing: Bottle of goolag, 10 sea urchins, a sack of orbelnuts and 32 graham crackers.",
        "Rules of the game: First one to make it to the other side of the island wins."
    ]
)

In [14]:
# Test vector database

collection.query(
    query_texts=["Travel time from Vandul to Farish?"],
    n_results=1
)

{'ids': [['id1']],
 'embeddings': None,
 'documents': [['Travel time: One-way flight from Vandul to Farish is 1 hour 30 minutes. One-way flight Equoza to Opinstian is 3 hours 45 minutes. One-way flight from Maddox to Lenivin is 8 hours.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None]],
 'distances': [[0.5448119640350342]]}

In [18]:
# Define function to fetch information from vector database

from mlflow.entities import Document

@mlflow.trace
def search_vector_db(query: str) -> tuple[str, str]:
    results = collection.query(
        query_texts=[query],
        n_results=1
    )
    return results["documents"][0][0], results["ids"][0]

@mlflow.trace(span_type=SpanType.RETRIEVER)
def retrieve_documents(query: str):
    """Retrieve documents from vector database."""

    doc, id = search_vector_db(query)

    span = mlflow.get_current_active_span()

    # mlflow specific format, will make logging easier
    outputs = [
        Document(page_content=doc, metadata={"id": id})
    ]

    span.set_outputs(outputs)

    # Return the original format for downstream usage
    return doc

In [ ]:
# Define langraph agent

from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

@tool
def get_gameshow_information(query: str) -> str:
    """Retrieve information about the gameshow."""
    return retrieve_documents(query)

model = "openai:gpt-4o-mini"
tools = [get_gameshow_information]
agent = create_react_agent(model=model, tools=tools)

In [27]:
agent.invoke({
    "messages": [{"role": "user", "content": "How do I win?"}]
})

2025/10/09 12:22:42 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/09 12:22:44 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/09 12:22:44 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/09 12:22:44 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/09 12:22:44 WARNING mlflow.tracing.export.mlflow_v3: Failed to log span to MLflow backend: INTERNAL_ERROR: Response: {'detail': 'REST OTLP span logging is not supported by FileStore'}
2025/10/09 12:22:44 WARNING mlflow.traci

{'messages': [HumanMessage(content='How do I win?', additional_kwargs={}, response_metadata={}, id='5bd25b69-08d0-4ae2-9868-e4feed5fc7e0'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_yncPucT84MujS8i5wRSkV6qP', 'function': {'arguments': '{"query":"How to win a gameshow"}', 'name': 'get_gameshow_information'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 51, 'total_tokens': 72, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_560af6e559', 'id': 'chatcmpl-COZedh99OQ6P1uA1nCBxL6DZa6Jyr', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--319961b3-c5f8-49d3-8132-4b6d2de41581-0', tool_calls=[{'name': 'get_gameshow_information', 'args': {'qu

Trace(trace_id=tr-0fcea48079485014ff4b0f37c4ade08a)